In [ ]:
# ============================================================
# ALEXNET + BAYESIAN OPT + 5-FOLD CV
# CV ON (TRAIN + VAL), TEST KEPT FIXED
# VANILLA LDM AUGMENTED DATASET
# ============================================================

import os, cv2
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report
import keras_tuner as kt

# ---------------- CONFIG ----------------
BASE_DIR = "/kaggle/input/datasets/pankajsthakre/VanillaLDM/BambooLeafDS"

TRAIN_DIR = os.path.join(BASE_DIR, "train")
VAL_DIR   = os.path.join(BASE_DIR, "val")
TEST_DIR  = os.path.join(BASE_DIR, "test")

IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 10
FOLDS = 5
SEED = 42

tf.random.set_seed(SEED)
np.random.seed(SEED)

# ---------------- LOAD FUNCTION ----------------
def load_paths_labels(directory):
    paths, labels = [], []
    class_names = sorted(os.listdir(directory))

    for i, cls in enumerate(class_names):
        cls_dir = os.path.join(directory, cls)
        for f in os.listdir(cls_dir):
            paths.append(os.path.join(cls_dir, f))
            labels.append(i)

    return np.array(paths), np.array(labels), class_names

# ---------------- LOAD DATA ----------------
X_train, y_train, class_names = load_paths_labels(TRAIN_DIR)
X_val, y_val, _ = load_paths_labels(VAL_DIR)
X_test, y_test, _ = load_paths_labels(TEST_DIR)

# 🔥 COMBINE TRAIN + VAL
X_full = np.concatenate([X_train, X_val])
y_full = np.concatenate([y_train, y_val])

NUM_CLASSES = len(class_names)

print("Train+Val:", len(X_full))
print("Test:", len(X_test))

# ---------------- GENERATOR ----------------
def generator(paths, labels):
    while True:
        idx = np.random.permutation(len(paths))

        for i in range(0, len(paths), BATCH_SIZE):
            batch = idx[i:i+BATCH_SIZE]
            imgs, labs = [], []

            for j in batch:
                img = cv2.imread(paths[j])
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
                img = img.astype("float32") / 255.0  # AlexNet normalization

                imgs.append(img)
                labs.append(labels[j])

            yield np.array(imgs), tf.keras.utils.to_categorical(labs, NUM_CLASSES)

# ============================================================
# 🔥 MODEL BUILDER (ALEXNET + BAYESIAN)
# ============================================================
def build_alexnet(hp):

    model = models.Sequential()

    # Conv Block 1
    model.add(layers.Conv2D(96, (11,11), strides=4, activation='relu',
                            input_shape=(224,224,3)))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D((3,3), strides=2))

    # Conv Block 2
    model.add(layers.Conv2D(256, (5,5), padding='same', activation='relu'))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D((3,3), strides=2))

    # Conv Blocks 3–5
    model.add(layers.Conv2D(384, (3,3), padding='same', activation='relu'))
    model.add(layers.Conv2D(384, (3,3), padding='same', activation='relu'))
    model.add(layers.Conv2D(256, (3,3), padding='same', activation='relu'))
    model.add(layers.MaxPooling2D((3,3), strides=2))

    model.add(layers.Flatten())

    # Tunable Dense Layers
    units = hp.Int("dense_units", 512, 2048, step=512)
    model.add(layers.Dense(units, activation='relu'))

    dropout = hp.Float("dropout", 0.3, 0.7, step=0.1)
    model.add(layers.Dropout(dropout))

    model.add(layers.Dense(units, activation='relu'))
    model.add(layers.Dropout(dropout))

    model.add(layers.Dense(NUM_CLASSES, activation='softmax'))

    # Tunable Learning Rate
    lr = hp.Choice("lr", [1e-3, 1e-4, 1e-5])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(lr),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    return model

# ============================================================
# 🔥 5-FOLD CV ON TRAIN+VAL
# ============================================================

skf = StratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=SEED)

fold_acc = []
best_hp_global = None
best_score = 0

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_full, y_full)):

    print(f"\n=========== FOLD {fold+1} ===========")

    X_tr, X_vl = X_full[tr_idx], X_full[val_idx]
    y_tr, y_vl = y_full[tr_idx], y_full[val_idx]

    tuner = kt.BayesianOptimization(
        build_alexnet,
        objective="val_accuracy",
        max_trials=5,
        directory="bayes_alexnet",
        project_name=f"fold_{fold+1}",
        overwrite=True
    )

    tuner.search(
        generator(X_tr, y_tr),
        steps_per_epoch=len(X_tr)//BATCH_SIZE,
        validation_data=generator(X_vl, y_vl),
        validation_steps=len(X_vl)//BATCH_SIZE,
        epochs=EPOCHS,
        verbose=1
    )

    best_hp = tuner.get_best_hyperparameters(1)[0]
    best_model = tuner.get_best_models(1)[0]

    # ---- validation evaluation ----
    y_true, y_pred = [], []
    val_gen = generator(X_vl, y_vl)

    for _ in range(len(X_vl)//BATCH_SIZE):
        xb, yb = next(val_gen)
        preds = best_model.predict(xb, verbose=0)

        y_true.extend(np.argmax(yb, axis=1))
        y_pred.extend(np.argmax(preds, axis=1))

    acc = accuracy_score(y_true, y_pred)
    fold_acc.append(acc)

    print(f"Fold {fold+1} Accuracy: {acc:.4f}")

    # Track best HP globally
    if acc > best_score:
        best_score = acc
        best_hp_global = best_hp

# ============================================================
# 🔥 FINAL TRAINING ON FULL (TRAIN+VAL)
# ============================================================

final_model = build_alexnet(best_hp_global)

final_model.fit(
    generator(X_full, y_full),
    steps_per_epoch=len(X_full)//BATCH_SIZE,
    epochs=EPOCHS,
    verbose=1
)

# ============================================================
# 🔥 TEST EVALUATION (UNCHANGED TEST SET)
# ============================================================

y_true, y_pred = [], []
test_gen = generator(X_test, y_test)

for _ in range(len(X_test)//BATCH_SIZE):
    xb, yb = next(test_gen)
    preds = final_model.predict(xb, verbose=0)

    y_true.extend(np.argmax(yb, axis=1))
    y_pred.extend(np.argmax(preds, axis=1))

print("\n🎯 FINAL TEST ACCURACY:", accuracy_score(y_true, y_pred))

print("\n📊 Classification Report:")
print(classification_report(y_true, y_pred, target_names=class_names))

# ============================================================
# 📊 CV RESULT
# ============================================================

print("\nCV Mean Accuracy:", np.mean(fold_acc))
print("CV Std:", np.std(fold_acc))

In [ ]:
# ============================================================
# VGG19 + BAYESIAN OPT + 5-FOLD CV
# CV ON (TRAIN + VAL), TEST KEPT FIXED
# VANILLA LDM AUGMENTED DATASET
# ============================================================

import os, cv2
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import VGG19
from tensorflow.keras.applications.vgg19 import preprocess_input
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report
import keras_tuner as kt

# ---------------- CONFIG ----------------
BASE_DIR = "/kaggle/input/datasets/pankajsthakre/VanillaLDM/BambooLeafDS"

TRAIN_DIR = os.path.join(BASE_DIR, "train")
VAL_DIR   = os.path.join(BASE_DIR, "val")
TEST_DIR  = os.path.join(BASE_DIR, "test")

IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 10
FOLDS = 5
SEED = 42

tf.random.set_seed(SEED)
np.random.seed(SEED)

# ---------------- LOAD FUNCTION ----------------
def load_paths_labels(directory):
    paths, labels = [], []
    class_names = sorted(os.listdir(directory))

    for i, cls in enumerate(class_names):
        cls_dir = os.path.join(directory, cls)
        for f in os.listdir(cls_dir):
            paths.append(os.path.join(cls_dir, f))
            labels.append(i)

    return np.array(paths), np.array(labels), class_names

# ---------------- LOAD DATA ----------------
X_train, y_train, class_names = load_paths_labels(TRAIN_DIR)
X_val, y_val, _ = load_paths_labels(VAL_DIR)
X_test, y_test, _ = load_paths_labels(TEST_DIR)

# 🔥 COMBINE TRAIN + VAL
X_full = np.concatenate([X_train, X_val])
y_full = np.concatenate([y_train, y_val])

NUM_CLASSES = len(class_names)

print("Train+Val:", len(X_full))
print("Test:", len(X_test))

# ---------------- GENERATOR ----------------
def generator(paths, labels):
    while True:
        idx = np.random.permutation(len(paths))

        for i in range(0, len(paths), BATCH_SIZE):
            batch = idx[i:i+BATCH_SIZE]
            imgs, labs = [], []

            for j in batch:
                img = cv2.imread(paths[j])
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
                img = preprocess_input(img.astype("float32"))

                imgs.append(img)
                labs.append(labels[j])

            yield np.array(imgs), tf.keras.utils.to_categorical(labs, NUM_CLASSES)

# ============================================================
# 🔥 MODEL BUILDER (VGG19 + BAYESIAN)
# ============================================================
def build_vgg19(hp):

    base = VGG19(
        weights="imagenet",
        include_top=False,
        input_shape=(224,224,3)
    )
    base.trainable = False

    x = layers.GlobalAveragePooling2D()(base.output)

    # Tunable Dense
    units = hp.Int("dense_units", 128, 512, step=128)
    x = layers.Dense(units, activation="relu")(x)

    # Tunable Dropout
    dropout = hp.Float("dropout", 0.3, 0.7, step=0.1)
    x = layers.Dropout(dropout)(x)

    outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)

    model = models.Model(base.input, outputs)

    # Tunable LR
    lr = hp.Choice("lr", [1e-3, 1e-4, 1e-5])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(lr),
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

# ============================================================
# 🔥 5-FOLD CV ON TRAIN+VAL
# ============================================================

skf = StratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=SEED)

fold_acc = []
best_hp_global = None
best_score = 0

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_full, y_full)):

    print(f"\n=========== FOLD {fold+1} ===========")

    X_tr, X_vl = X_full[tr_idx], X_full[val_idx]
    y_tr, y_vl = y_full[tr_idx], y_full[val_idx]

    tuner = kt.BayesianOptimization(
        build_vgg19,
        objective="val_accuracy",
        max_trials=5,
        directory="bayes_vgg19",
        project_name=f"fold_{fold+1}",
        overwrite=True
    )

    tuner.search(
        generator(X_tr, y_tr),
        steps_per_epoch=len(X_tr)//BATCH_SIZE,
        validation_data=generator(X_vl, y_vl),
        validation_steps=len(X_vl)//BATCH_SIZE,
        epochs=EPOCHS,
        verbose=1
    )

    best_hp = tuner.get_best_hyperparameters(1)[0]
    best_model = tuner.get_best_models(1)[0]

    # ---- validation evaluation ----
    y_true, y_pred = [], []
    val_gen = generator(X_vl, y_vl)

    for _ in range(len(X_vl)//BATCH_SIZE):
        xb, yb = next(val_gen)
        preds = best_model.predict(xb, verbose=0)

        y_true.extend(np.argmax(yb, axis=1))
        y_pred.extend(np.argmax(preds, axis=1))

    acc = accuracy_score(y_true, y_pred)
    fold_acc.append(acc)

    print(f"Fold {fold+1} Accuracy: {acc:.4f}")

    # Track best HP globally
    if acc > best_score:
        best_score = acc
        best_hp_global = best_hp

# ============================================================
# 🔥 FINAL TRAINING ON FULL (TRAIN+VAL)
# ============================================================

final_model = build_vgg19(best_hp_global)

final_model.fit(
    generator(X_full, y_full),
    steps_per_epoch=len(X_full)//BATCH_SIZE,
    epochs=EPOCHS,
    verbose=1
)

# ============================================================
# 🔥 TEST EVALUATION (UNCHANGED TEST SET)
# ============================================================

y_true, y_pred = [], []
test_gen = generator(X_test, y_test)

for _ in range(len(X_test)//BATCH_SIZE):
    xb, yb = next(test_gen)
    preds = final_model.predict(xb, verbose=0)

    y_true.extend(np.argmax(yb, axis=1))
    y_pred.extend(np.argmax(preds, axis=1))

print("\n🎯 FINAL TEST ACCURACY:", accuracy_score(y_true, y_pred))

print("\n📊 Classification Report:")
print(classification_report(y_true, y_pred, target_names=class_names))

# ============================================================
# 📊 CV RESULT
# ============================================================

print("\nCV Mean Accuracy:", np.mean(fold_acc))
print("CV Std:", np.std(fold_acc))

In [ ]:
# ============================================================
# RESNET50 + BAYESIAN OPT + 5-FOLD CV
# CV ON (TRAIN + VAL), TEST KEPT FIXED
# VANILLA LDM AUGMENTED DATASET
# ============================================================

import os, cv2
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report
import keras_tuner as kt

# ---------------- CONFIG ----------------
BASE_DIR = "/kaggle/input/datasets/pankajsthakre/VanillaLDM/BambooLeafDS"

TRAIN_DIR = os.path.join(BASE_DIR, "train")
VAL_DIR   = os.path.join(BASE_DIR, "val")
TEST_DIR  = os.path.join(BASE_DIR, "test")

IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 10
FOLDS = 5
SEED = 42

tf.random.set_seed(SEED)
np.random.seed(SEED)

# ---------------- LOAD FUNCTION ----------------
def load_paths_labels(directory):
    paths, labels = [], []
    class_names = sorted(os.listdir(directory))

    for i, cls in enumerate(class_names):
        cls_dir = os.path.join(directory, cls)
        for f in os.listdir(cls_dir):
            paths.append(os.path.join(cls_dir, f))
            labels.append(i)

    return np.array(paths), np.array(labels), class_names

# ---------------- LOAD DATA ----------------
X_train, y_train, class_names = load_paths_labels(TRAIN_DIR)
X_val, y_val, _ = load_paths_labels(VAL_DIR)
X_test, y_test, _ = load_paths_labels(TEST_DIR)

# 🔥 COMBINE TRAIN + VAL
X_full = np.concatenate([X_train, X_val])
y_full = np.concatenate([y_train, y_val])

NUM_CLASSES = len(class_names)

print("Train+Val:", len(X_full))
print("Test:", len(X_test))

# ---------------- GENERATOR ----------------
def generator(paths, labels):
    while True:
        idx = np.random.permutation(len(paths))

        for i in range(0, len(paths), BATCH_SIZE):
            batch = idx[i:i+BATCH_SIZE]
            imgs, labs = [], []

            for j in batch:
                img = cv2.imread(paths[j])
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                img = cv2.resize(img, (224,224))
                img = preprocess_input(img.astype("float32"))

                imgs.append(img)
                labs.append(labels[j])

            yield np.array(imgs), tf.keras.utils.to_categorical(labs, NUM_CLASSES)

# ============================================================
# 🔥 MODEL BUILDER
# ============================================================
def build_resnet50(hp):

    base = ResNet50(weights="imagenet", include_top=False,
                    input_shape=(224,224,3))
    base.trainable = False

    x = layers.GlobalAveragePooling2D()(base.output)

    units = hp.Int("dense_units", 128, 512, step=128)
    x = layers.Dense(units, activation="relu")(x)

    dropout = hp.Float("dropout", 0.3, 0.7, step=0.1)
    x = layers.Dropout(dropout)(x)

    outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)

    model = models.Model(base.input, outputs)

    lr = hp.Choice("lr", [1e-3, 1e-4, 1e-5])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(lr),
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

# ============================================================
# 🔥 5-FOLD CV ON TRAIN+VAL
# ============================================================

skf = StratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=SEED)

fold_acc = []
best_hp_global = None
best_score = 0

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_full, y_full)):

    print(f"\n=========== FOLD {fold+1} ===========")

    X_tr, X_vl = X_full[tr_idx], X_full[val_idx]
    y_tr, y_vl = y_full[tr_idx], y_full[val_idx]

    tuner = kt.BayesianOptimization(
        build_resnet50,
        objective="val_accuracy",
        max_trials=5,
        directory="bayes_resnet50",
        project_name=f"fold_{fold+1}",
        overwrite=True
    )

    tuner.search(
        generator(X_tr, y_tr),
        steps_per_epoch=len(X_tr)//BATCH_SIZE,
        validation_data=generator(X_vl, y_vl),
        validation_steps=len(X_vl)//BATCH_SIZE,
        epochs=EPOCHS,
        verbose=1
    )

    best_hp = tuner.get_best_hyperparameters(1)[0]
    best_model = tuner.get_best_models(1)[0]

    # ---- validation evaluation ----
    y_true, y_pred = [], []
    val_gen = generator(X_vl, y_vl)

    for _ in range(len(X_vl)//BATCH_SIZE):
        xb, yb = next(val_gen)
        preds = best_model.predict(xb, verbose=0)

        y_true.extend(np.argmax(yb, axis=1))
        y_pred.extend(np.argmax(preds, axis=1))

    acc = accuracy_score(y_true, y_pred)
    fold_acc.append(acc)

    print(f"Fold {fold+1} Accuracy: {acc:.4f}")

    # Track best HP globally
    if acc > best_score:
        best_score = acc
        best_hp_global = best_hp

# ============================================================
# 🔥 FINAL TRAINING ON FULL (TRAIN+VAL)
# ============================================================

final_model = build_resnet50(best_hp_global)

final_model.fit(
    generator(X_full, y_full),
    steps_per_epoch=len(X_full)//BATCH_SIZE,
    epochs=EPOCHS,
    verbose=1
)

# ============================================================
# 🔥 TEST EVALUATION (UNCHANGED TEST SET)
# ============================================================

y_true, y_pred = [], []
test_gen = generator(X_test, y_test)

for _ in range(len(X_test)//BATCH_SIZE):
    xb, yb = next(test_gen)
    preds = final_model.predict(xb, verbose=0)

    y_true.extend(np.argmax(yb, axis=1))
    y_pred.extend(np.argmax(preds, axis=1))

print("\n🎯 FINAL TEST ACCURACY:", accuracy_score(y_true, y_pred))
print("\n📊 Classification Report:")
print(classification_report(y_true, y_pred, target_names=class_names))

# ============================================================
# 📊 CV RESULT
# ============================================================

print("\nCV Mean Accuracy:", np.mean(fold_acc))
print("CV Std:", np.std(fold_acc))

In [ ]:

# ============================================================
# ViT-B16 + BAYESIAN OPT + 5-FOLD CV
# CV ON (TRAIN + VAL), TEST KEPT FIXED
# VANILLA LDM AUGMENTED DATASET
# ============================================================
!pip install vit-keras
import os, cv2
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report
import keras_tuner as kt

from vit_keras import vit

# ---------------- CONFIG ----------------
BASE_DIR = "/kaggle/input/datasets/pankajsthakre/VanillaLDM/BambooLeafDS"

TRAIN_DIR = os.path.join(BASE_DIR, "train")
VAL_DIR   = os.path.join(BASE_DIR, "val")
TEST_DIR  = os.path.join(BASE_DIR, "test")

IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 10
FOLDS = 5
SEED = 42

tf.random.set_seed(SEED)
np.random.seed(SEED)

# ---------------- LOAD FUNCTION ----------------
def load_paths_labels(directory):
    paths, labels = [], []
    class_names = sorted(os.listdir(directory))

    for i, cls in enumerate(class_names):
        cls_dir = os.path.join(directory, cls)
        for f in os.listdir(cls_dir):
            paths.append(os.path.join(cls_dir, f))
            labels.append(i)

    return np.array(paths), np.array(labels), class_names

# ---------------- LOAD DATA ----------------
X_train, y_train, class_names = load_paths_labels(TRAIN_DIR)
X_val, y_val, _ = load_paths_labels(VAL_DIR)
X_test, y_test, _ = load_paths_labels(TEST_DIR)

# 🔥 COMBINE TRAIN + VAL
X_full = np.concatenate([X_train, X_val])
y_full = np.concatenate([y_train, y_val])

NUM_CLASSES = len(class_names)

print("Train+Val:", len(X_full))
print("Test:", len(X_test))

# ---------------- GENERATOR ----------------
def generator(paths, labels):
    while True:
        idx = np.random.permutation(len(paths))

        for i in range(0, len(paths), BATCH_SIZE):
            batch = idx[i:i+BATCH_SIZE]
            imgs, labs = [], []

            for j in batch:
                img = cv2.imread(paths[j])
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
                img = img.astype("float32") / 255.0   # ViT normalization

                imgs.append(img)
                labs.append(labels[j])

            yield np.array(imgs), tf.keras.utils.to_categorical(labs, NUM_CLASSES)

# ============================================================
# 🔥 MODEL BUILDER (ViT-B16 + BAYESIAN)
# ============================================================
def build_vit(hp):

    vit_model = vit.vit_b16(
        image_size=224,
        pretrained=True,
        include_top=False,
        pretrained_top=False
    )

    vit_model.trainable = False  # Freeze backbone

    x = layers.Flatten()(vit_model.output)

    # Tunable Dense
    units = hp.Int("dense_units", 128, 512, step=128)
    x = layers.Dense(units, activation="relu")(x)

    # Tunable Dropout
    dropout = hp.Float("dropout", 0.3, 0.7, step=0.1)
    x = layers.Dropout(dropout)(x)

    outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)

    model = models.Model(vit_model.input, outputs)

    # Tunable LR
    lr = hp.Choice("lr", [1e-3, 1e-4, 1e-5])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(lr),
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

# ============================================================
# 🔥 5-FOLD CV ON TRAIN+VAL
# ============================================================

skf = StratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=SEED)

fold_acc = []
best_hp_global = None
best_score = 0

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_full, y_full)):

    print(f"\n=========== FOLD {fold+1} ===========")

    X_tr, X_vl = X_full[tr_idx], X_full[val_idx]
    y_tr, y_vl = y_full[tr_idx], y_full[val_idx]

    tuner = kt.BayesianOptimization(
        build_vit,
        objective="val_accuracy",
        max_trials=5,
        directory="bayes_vitb16",
        project_name=f"fold_{fold+1}",
        overwrite=True
    )

    tuner.search(
        generator(X_tr, y_tr),
        steps_per_epoch=len(X_tr)//BATCH_SIZE,
        validation_data=generator(X_vl, y_vl),
        validation_steps=len(X_vl)//BATCH_SIZE,
        epochs=EPOCHS,
        verbose=1
    )

    best_hp = tuner.get_best_hyperparameters(1)[0]
    best_model = tuner.get_best_models(1)[0]

    # ---- validation evaluation ----
    y_true, y_pred = [], []
    val_gen = generator(X_vl, y_vl)

    for _ in range(len(X_vl)//BATCH_SIZE):
        xb, yb = next(val_gen)
        preds = best_model.predict(xb, verbose=0)

        y_true.extend(np.argmax(yb, axis=1))
        y_pred.extend(np.argmax(preds, axis=1))

    acc = accuracy_score(y_true, y_pred)
    fold_acc.append(acc)

    print(f"Fold {fold+1} Accuracy: {acc:.4f}")

    # Track best HP globally
    if acc > best_score:
        best_score = acc
        best_hp_global = best_hp

# ============================================================
# 🔥 FINAL TRAINING ON FULL (TRAIN+VAL)
# ============================================================

final_model = build_vit(best_hp_global)

final_model.fit(
    generator(X_full, y_full),
    steps_per_epoch=len(X_full)//BATCH_SIZE,
    epochs=EPOCHS,
    verbose=1
)

# ============================================================
# 🔥 TEST EVALUATION (UNCHANGED TEST SET)
# ============================================================

y_true, y_pred = [], []
test_gen = generator(X_test, y_test)

for _ in range(len(X_test)//BATCH_SIZE):
    xb, yb = next(test_gen)
    preds = final_model.predict(xb, verbose=0)

    y_true.extend(np.argmax(yb, axis=1))
    y_pred.extend(np.argmax(preds, axis=1))

print("\n🎯 FINAL TEST ACCURACY:", accuracy_score(y_true, y_pred))

print("\n📊 Classification Report:")
print(classification_report(y_true, y_pred, target_names=class_names))

# ============================================================
# 📊 CV RESULT
# ============================================================

print("\nCV Mean Accuracy:", np.mean(fold_acc))
print("CV Std:", np.std(fold_acc))

In [ ]:
# ============================================================
# DENSENET201 + BAYESIAN OPT + 5-FOLD CV
# CV ON (TRAIN + VAL), TEST KEPT FIXED
# VANILLA LDM AUGMENTED DATASET
# ============================================================

import os, cv2
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import DenseNet201
from tensorflow.keras.applications.densenet import preprocess_input
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report
import keras_tuner as kt

# ---------------- CONFIG ----------------
BASE_DIR = "/kaggle/input/datasets/pankajsthakre/VaillaLDM/BambooLeafDS"

TRAIN_DIR = os.path.join(BASE_DIR, "train")
VAL_DIR   = os.path.join(BASE_DIR, "val")
TEST_DIR  = os.path.join(BASE_DIR, "test")

IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 10
FOLDS = 5
SEED = 42

tf.random.set_seed(SEED)
np.random.seed(SEED)

# ---------------- LOAD FUNCTION ----------------
def load_paths_labels(directory):
    paths, labels = [], []
    class_names = sorted(os.listdir(directory))

    for i, cls in enumerate(class_names):
        cls_dir = os.path.join(directory, cls)
        for f in os.listdir(cls_dir):
            paths.append(os.path.join(cls_dir, f))
            labels.append(i)

    return np.array(paths), np.array(labels), class_names

# ---------------- LOAD DATA ----------------
X_train, y_train, class_names = load_paths_labels(TRAIN_DIR)
X_val, y_val, _ = load_paths_labels(VAL_DIR)
X_test, y_test, _ = load_paths_labels(TEST_DIR)

# 🔥 COMBINE TRAIN + VAL
X_full = np.concatenate([X_train, X_val])
y_full = np.concatenate([y_train, y_val])

NUM_CLASSES = len(class_names)

print("Train+Val:", len(X_full))
print("Test:", len(X_test))

# ---------------- GENERATOR ----------------
def generator(paths, labels):
    while True:
        idx = np.random.permutation(len(paths))

        for i in range(0, len(paths), BATCH_SIZE):
            batch = idx[i:i+BATCH_SIZE]
            imgs, labs = [], []

            for j in batch:
                img = cv2.imread(paths[j])
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
                img = preprocess_input(img.astype("float32"))

                imgs.append(img)
                labs.append(labels[j])

            yield np.array(imgs), tf.keras.utils.to_categorical(labs, NUM_CLASSES)

# ============================================================
# 🔥 MODEL BUILDER (DENSENET201 + BAYESIAN)
# ============================================================
def build_densenet201(hp):

    base = DenseNet201(
        weights="imagenet",
        include_top=False,
        input_shape=(224,224,3)
    )
    base.trainable = False  # Freeze backbone

    x = layers.GlobalAveragePooling2D()(base.output)

    # Tunable Dense Layer
    units = hp.Int("dense_units", 128, 512, step=128)
    x = layers.Dense(units, activation="relu")(x)

    # Tunable Dropout
    dropout = hp.Float("dropout", 0.3, 0.7, step=0.1)
    x = layers.Dropout(dropout)(x)

    outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)

    model = models.Model(base.input, outputs)

    # Tunable Learning Rate
    lr = hp.Choice("lr", [1e-3, 1e-4, 1e-5])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(lr),
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

# ============================================================
# 🔥 5-FOLD CV ON TRAIN+VAL
# ============================================================

skf = StratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=SEED)

fold_acc = []
best_hp_global = None
best_score = 0

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_full, y_full)):

    print(f"\n=========== FOLD {fold+1} ===========")

    X_tr, X_vl = X_full[tr_idx], X_full[val_idx]
    y_tr, y_vl = y_full[tr_idx], y_full[val_idx]

    tuner = kt.BayesianOptimization(
        build_densenet201,
        objective="val_accuracy",
        max_trials=5,
        directory="bayes_densenet201",
        project_name=f"fold_{fold+1}",
        overwrite=True
    )

    tuner.search(
        generator(X_tr, y_tr),
        steps_per_epoch=len(X_tr)//BATCH_SIZE,
        validation_data=generator(X_vl, y_vl),
        validation_steps=len(X_vl)//BATCH_SIZE,
        epochs=EPOCHS,
        verbose=1
    )

    best_hp = tuner.get_best_hyperparameters(1)[0]
    best_model = tuner.get_best_models(1)[0]

    # ---- validation evaluation ----
    y_true, y_pred = [], []
    val_gen = generator(X_vl, y_vl)

    for _ in range(len(X_vl)//BATCH_SIZE):
        xb, yb = next(val_gen)
        preds = best_model.predict(xb, verbose=0)

        y_true.extend(np.argmax(yb, axis=1))
        y_pred.extend(np.argmax(preds, axis=1))

    acc = accuracy_score(y_true, y_pred)
    fold_acc.append(acc)

    print(f"Fold {fold+1} Accuracy: {acc:.4f}")

    # Track best HP globally
    if acc > best_score:
        best_score = acc
        best_hp_global = best_hp

# ============================================================
# 🔥 FINAL TRAINING ON FULL (TRAIN+VAL)
# ============================================================

final_model = build_densenet201(best_hp_global)

final_model.fit(
    generator(X_full, y_full),
    steps_per_epoch=len(X_full)//BATCH_SIZE,
    epochs=EPOCHS,
    verbose=1
)

# ============================================================
# 🔥 TEST EVALUATION (UNCHANGED TEST SET)
# ============================================================

y_true, y_pred = [], []
test_gen = generator(X_test, y_test)

for _ in range(len(X_test)//BATCH_SIZE):
    xb, yb = next(test_gen)
    preds = final_model.predict(xb, verbose=0)

    y_true.extend(np.argmax(yb, axis=1))
    y_pred.extend(np.argmax(preds, axis=1))

print("\n🎯 FINAL TEST ACCURACY:", accuracy_score(y_true, y_pred))

print("\n📊 Classification Report:")
print(classification_report(y_true, y_pred, target_names=class_names))

# ============================================================
# 📊 CV RESULT
# ============================================================

print("\nCV Mean Accuracy:", np.mean(fold_acc))
print("CV Std:", np.std(fold_acc))

In [ ]:
# ============================================================
# EFFICIENTNETB0 + BAYESIAN OPT + 5-FOLD CV
# CV ON (TRAIN + VAL), TEST KEPT FIXED
# VANILLA LDM AUGMENTED DATASET
# ============================================================

import os, cv2
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report
import keras_tuner as kt

# ---------------- CONFIG ----------------
BASE_DIR = "/kaggle/input/datasets/pankajsthakre/VanillaLDM/BambooLeafDS"

TRAIN_DIR = os.path.join(BASE_DIR, "train")
VAL_DIR   = os.path.join(BASE_DIR, "val")
TEST_DIR  = os.path.join(BASE_DIR, "test")

IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 10
FOLDS = 5
SEED = 42

tf.random.set_seed(SEED)
np.random.seed(SEED)

# ---------------- LOAD FUNCTION ----------------
def load_paths_labels(directory):
    paths, labels = [], []
    class_names = sorted(os.listdir(directory))

    for i, cls in enumerate(class_names):
        cls_dir = os.path.join(directory, cls)
        for f in os.listdir(cls_dir):
            paths.append(os.path.join(cls_dir, f))
            labels.append(i)

    return np.array(paths), np.array(labels), class_names

# ---------------- LOAD DATA ----------------
X_train, y_train, class_names = load_paths_labels(TRAIN_DIR)
X_val, y_val, _ = load_paths_labels(VAL_DIR)
X_test, y_test, _ = load_paths_labels(TEST_DIR)

# 🔥 COMBINE TRAIN + VAL
X_full = np.concatenate([X_train, X_val])
y_full = np.concatenate([y_train, y_val])

NUM_CLASSES = len(class_names)

print("Train+Val:", len(X_full))
print("Test:", len(X_test))

# ---------------- GENERATOR ----------------
def generator(paths, labels):
    while True:
        idx = np.random.permutation(len(paths))

        for i in range(0, len(paths), BATCH_SIZE):
            batch = idx[i:i+BATCH_SIZE]
            imgs, labs = [], []

            for j in batch:
                img = cv2.imread(paths[j])
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
                img = preprocess_input(img.astype("float32"))

                imgs.append(img)
                labs.append(labels[j])

            yield np.array(imgs), tf.keras.utils.to_categorical(labs, NUM_CLASSES)

# ============================================================
# 🔥 MODEL BUILDER (EFFICIENTNETB0 + BAYESIAN)
# ============================================================
def build_efficientnetb0(hp):

    base = EfficientNetB0(
        weights="imagenet",
        include_top=False,
        input_shape=(224,224,3)
    )
    base.trainable = False  # Freeze backbone

    x = layers.GlobalAveragePooling2D()(base.output)

    # Tunable Dense
    units = hp.Int("dense_units", 128, 512, step=128)
    x = layers.Dense(units, activation="relu")(x)

    # Tunable Dropout
    dropout = hp.Float("dropout", 0.3, 0.7, step=0.1)
    x = layers.Dropout(dropout)(x)

    outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)

    model = models.Model(base.input, outputs)

    # Tunable LR
    lr = hp.Choice("lr", [1e-3, 1e-4, 1e-5])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(lr),
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

# ============================================================
# 🔥 5-FOLD CV ON TRAIN+VAL
# ============================================================

skf = StratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=SEED)

fold_acc = []
best_hp_global = None
best_score = 0

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_full, y_full)):

    print(f"\n=========== FOLD {fold+1} ===========")

    X_tr, X_vl = X_full[tr_idx], X_full[val_idx]
    y_tr, y_vl = y_full[tr_idx], y_full[val_idx]

    tuner = kt.BayesianOptimization(
        build_efficientnetb0,
        objective="val_accuracy",
        max_trials=5,
        directory="bayes_efficientnetb0",
        project_name=f"fold_{fold+1}",
        overwrite=True
    )

    tuner.search(
        generator(X_tr, y_tr),
        steps_per_epoch=len(X_tr)//BATCH_SIZE,
        validation_data=generator(X_vl, y_vl),
        validation_steps=len(X_vl)//BATCH_SIZE,
        epochs=EPOCHS,
        verbose=1
    )

    best_hp = tuner.get_best_hyperparameters(1)[0]
    best_model = tuner.get_best_models(1)[0]

    # ---- validation evaluation ----
    y_true, y_pred = [], []
    val_gen = generator(X_vl, y_vl)

    for _ in range(len(X_vl)//BATCH_SIZE):
        xb, yb = next(val_gen)
        preds = best_model.predict(xb, verbose=0)

        y_true.extend(np.argmax(yb, axis=1))
        y_pred.extend(np.argmax(preds, axis=1))

    acc = accuracy_score(y_true, y_pred)
    fold_acc.append(acc)

    print(f"Fold {fold+1} Accuracy: {acc:.4f}")

    # Track best HP globally
    if acc > best_score:
        best_score = acc
        best_hp_global = best_hp

# ============================================================
# 🔥 FINAL TRAINING ON FULL (TRAIN+VAL)
# ============================================================

final_model = build_efficientnetb0(best_hp_global)

final_model.fit(
    generator(X_full, y_full),
    steps_per_epoch=len(X_full)//BATCH_SIZE,
    epochs=EPOCHS,
    verbose=1
)

# ============================================================
# 🔥 TEST EVALUATION (UNCHANGED TEST SET)
# ============================================================

y_true, y_pred = [], []
test_gen = generator(X_test, y_test)

for _ in range(len(X_test)//BATCH_SIZE):
    xb, yb = next(test_gen)
    preds = final_model.predict(xb, verbose=0)

    y_true.extend(np.argmax(yb, axis=1))
    y_pred.extend(np.argmax(preds, axis=1))

print("\n🎯 FINAL TEST ACCURACY:", accuracy_score(y_true, y_pred))

print("\n📊 Classification Report:")
print(classification_report(y_true, y_pred, target_names=class_names))

# ============================================================
# 📊 CV RESULT
# ============================================================

print("\nCV Mean Accuracy:", np.mean(fold_acc))
print("CV Std:", np.std(fold_acc))

In [ ]:
# ============================================================
# INCEPTIONV4 + BAYESIAN OPT + 5-FOLD CV
# CV ON (TRAIN + VAL), TEST KEPT FIXED
# VANILLA LDM AUGMENTED DATASET
# ============================================================
!pip install keras-applications
import os, cv2
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report
import keras_tuner as kt

# InceptionV4 (custom)
from keras_applications.inception_v4 import InceptionV4, preprocess_input

# ---------------- CONFIG ----------------
BASE_DIR = "/kaggle/input/datasets/pankajsthakre/VanillaLDM/BambooLeafDS"

TRAIN_DIR = os.path.join(BASE_DIR, "train")
VAL_DIR   = os.path.join(BASE_DIR, "val")
TEST_DIR  = os.path.join(BASE_DIR, "test")

IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 10
FOLDS = 5
SEED = 42

tf.random.set_seed(SEED)
np.random.seed(SEED)

# ---------------- LOAD FUNCTION ----------------
def load_paths_labels(directory):
    paths, labels = [], []
    class_names = sorted(os.listdir(directory))

    for i, cls in enumerate(class_names):
        cls_dir = os.path.join(directory, cls)
        for f in os.listdir(cls_dir):
            paths.append(os.path.join(cls_dir, f))
            labels.append(i)

    return np.array(paths), np.array(labels), class_names

# ---------------- LOAD DATA ----------------
X_train, y_train, class_names = load_paths_labels(TRAIN_DIR)
X_val, y_val, _ = load_paths_labels(VAL_DIR)
X_test, y_test, _ = load_paths_labels(TEST_DIR)

# 🔥 COMBINE TRAIN + VAL
X_full = np.concatenate([X_train, X_val])
y_full = np.concatenate([y_train, y_val])

NUM_CLASSES = len(class_names)

print("Train+Val:", len(X_full))
print("Test:", len(X_test))

# ---------------- GENERATOR ----------------
def generator(paths, labels):
    while True:
        idx = np.random.permutation(len(paths))

        for i in range(0, len(paths), BATCH_SIZE):
            batch = idx[i:i+BATCH_SIZE]
            imgs, labs = [], []

            for j in batch:
                img = cv2.imread(paths[j])
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
                img = preprocess_input(img.astype("float32"))

                imgs.append(img)
                labs.append(labels[j])

            yield np.array(imgs), tf.keras.utils.to_categorical(labs, NUM_CLASSES)

# ============================================================
# 🔥 MODEL BUILDER (INCEPTIONV4 + BAYESIAN)
# ============================================================
def build_inceptionv4(hp):

    base = InceptionV4(
        include_top=False,
        weights="imagenet",
        input_shape=(224,224,3)
    )
    base.trainable = False

    x = layers.GlobalAveragePooling2D()(base.output)

    # Tunable Dense
    units = hp.Int("dense_units", 128, 512, step=128)
    x = layers.Dense(units, activation="relu")(x)

    # Tunable Dropout
    dropout = hp.Float("dropout", 0.3, 0.7, step=0.1)
    x = layers.Dropout(dropout)(x)

    outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)

    model = models.Model(base.input, outputs)

    # Tunable LR
    lr = hp.Choice("lr", [1e-3, 1e-4, 1e-5])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(lr),
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

# ============================================================
# 🔥 5-FOLD CV ON TRAIN+VAL
# ============================================================

skf = StratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=SEED)

fold_acc = []
best_hp_global = None
best_score = 0

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_full, y_full)):

    print(f"\n=========== FOLD {fold+1} ===========")

    X_tr, X_vl = X_full[tr_idx], X_full[val_idx]
    y_tr, y_vl = y_full[tr_idx], y_full[val_idx]

    tuner = kt.BayesianOptimization(
        build_inceptionv4,
        objective="val_accuracy",
        max_trials=5,
        directory="bayes_inceptionv4",
        project_name=f"fold_{fold+1}",
        overwrite=True
    )

    tuner.search(
        generator(X_tr, y_tr),
        steps_per_epoch=len(X_tr)//BATCH_SIZE,
        validation_data=generator(X_vl, y_vl),
        validation_steps=len(X_vl)//BATCH_SIZE,
        epochs=EPOCHS,
        verbose=1
    )

    best_hp = tuner.get_best_hyperparameters(1)[0]
    best_model = tuner.get_best_models(1)[0]

    # ---- validation evaluation ----
    y_true, y_pred = [], []
    val_gen = generator(X_vl, y_vl)

    for _ in range(len(X_vl)//BATCH_SIZE):
        xb, yb = next(val_gen)
        preds = best_model.predict(xb, verbose=0)

        y_true.extend(np.argmax(yb, axis=1))
        y_pred.extend(np.argmax(preds, axis=1))

    acc = accuracy_score(y_true, y_pred)
    fold_acc.append(acc)

    print(f"Fold {fold+1} Accuracy: {acc:.4f}")

    # Track best HP globally
    if acc > best_score:
        best_score = acc
        best_hp_global = best_hp

# ============================================================
# 🔥 FINAL TRAINING ON FULL (TRAIN+VAL)
# ============================================================

final_model = build_inceptionv4(best_hp_global)

final_model.fit(
    generator(X_full, y_full),
    steps_per_epoch=len(X_full)//BATCH_SIZE,
    epochs=EPOCHS,
    verbose=1
)

# ============================================================
# 🔥 TEST EVALUATION (UNCHANGED TEST SET)
# ============================================================

y_true, y_pred = [], []
test_gen = generator(X_test, y_test)

for _ in range(len(X_test)//BATCH_SIZE):
    xb, yb = next(test_gen)
    preds = final_model.predict(xb, verbose=0)

    y_true.extend(np.argmax(yb, axis=1))
    y_pred.extend(np.argmax(preds, axis=1))

print("\n🎯 FINAL TEST ACCURACY:", accuracy_score(y_true, y_pred))

print("\n📊 Classification Report:")
print(classification_report(y_true, y_pred, target_names=class_names))

# ============================================================
# 📊 CV RESULT
# ============================================================

print("\nCV Mean Accuracy:", np.mean(fold_acc))
print("CV Std:", np.std(fold_acc))

In [ ]:
# ============================================================
# MOBILENETV1 + BAYESIAN OPT + 5-FOLD CV
# CV ON (TRAIN + VAL), TEST KEPT FIXED
# VANILLA LDM AUGMENTED DATASET
# ============================================================

import os, cv2
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNet
from tensorflow.keras.applications.mobilenet import preprocess_input
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report
import keras_tuner as kt

# ---------------- CONFIG ----------------
BASE_DIR = "/kaggle/input/datasets/pankajsthakre/VanillaLDM/BambooLeafDS"

TRAIN_DIR = os.path.join(BASE_DIR, "train")
VAL_DIR   = os.path.join(BASE_DIR, "val")
TEST_DIR  = os.path.join(BASE_DIR, "test")

IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 10
FOLDS = 5
SEED = 42

tf.random.set_seed(SEED)
np.random.seed(SEED)

# ---------------- LOAD FUNCTION ----------------
def load_paths_labels(directory):
    paths, labels = [], []
    class_names = sorted(os.listdir(directory))

    for i, cls in enumerate(class_names):
        cls_dir = os.path.join(directory, cls)
        for f in os.listdir(cls_dir):
            paths.append(os.path.join(cls_dir, f))
            labels.append(i)

    return np.array(paths), np.array(labels), class_names

# ---------------- LOAD DATA ----------------
X_train, y_train, class_names = load_paths_labels(TRAIN_DIR)
X_val, y_val, _ = load_paths_labels(VAL_DIR)
X_test, y_test, _ = load_paths_labels(TEST_DIR)

# 🔥 COMBINE TRAIN + VAL
X_full = np.concatenate([X_train, X_val])
y_full = np.concatenate([y_train, y_val])

NUM_CLASSES = len(class_names)

print("Train+Val:", len(X_full))
print("Test:", len(X_test))

# ---------------- GENERATOR ----------------
def generator(paths, labels):
    while True:
        idx = np.random.permutation(len(paths))

        for i in range(0, len(paths), BATCH_SIZE):
            batch = idx[i:i+BATCH_SIZE]
            imgs, labs = [], []

            for j in batch:
                img = cv2.imread(paths[j])
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
                img = preprocess_input(img.astype("float32"))

                imgs.append(img)
                labs.append(labels[j])

            yield np.array(imgs), tf.keras.utils.to_categorical(labs, NUM_CLASSES)

# ============================================================
# 🔥 MODEL BUILDER (MobileNetV1 + BAYESIAN)
# ============================================================
def build_mobilenet(hp):

    base = MobileNet(
        weights="imagenet",
        include_top=False,
        input_shape=(224,224,3)
    )
    base.trainable = False  # freeze backbone

    x = layers.GlobalAveragePooling2D()(base.output)

    # Tunable Dense
    units = hp.Int("dense_units", 128, 512, step=128)
    x = layers.Dense(units, activation="relu")(x)

    # Tunable Dropout
    dropout = hp.Float("dropout", 0.3, 0.7, step=0.1)
    x = layers.Dropout(dropout)(x)

    outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)

    model = models.Model(base.input, outputs)

    # Tunable LR
    lr = hp.Choice("lr", [1e-3, 1e-4, 1e-5])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(lr),
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

# ============================================================
# 🔥 5-FOLD CV ON TRAIN+VAL
# ============================================================

skf = StratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=SEED)

fold_acc = []
best_hp_global = None
best_score = 0

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_full, y_full)):

    print(f"\n=========== FOLD {fold+1} ===========")

    X_tr, X_vl = X_full[tr_idx], X_full[val_idx]
    y_tr, y_vl = y_full[tr_idx], y_full[val_idx]

    tuner = kt.BayesianOptimization(
        build_mobilenet,
        objective="val_accuracy",
        max_trials=5,
        directory="bayes_mobilenet",
        project_name=f"fold_{fold+1}",
        overwrite=True
    )

    tuner.search(
        generator(X_tr, y_tr),
        steps_per_epoch=len(X_tr)//BATCH_SIZE,
        validation_data=generator(X_vl, y_vl),
        validation_steps=len(X_vl)//BATCH_SIZE,
        epochs=EPOCHS,
        verbose=1
    )

    best_hp = tuner.get_best_hyperparameters(1)[0]
    best_model = tuner.get_best_models(1)[0]

    # ---- validation evaluation ----
    y_true, y_pred = [], []
    val_gen = generator(X_vl, y_vl)

    for _ in range(len(X_vl)//BATCH_SIZE):
        xb, yb = next(val_gen)
        preds = best_model.predict(xb, verbose=0)

        y_true.extend(np.argmax(yb, axis=1))
        y_pred.extend(np.argmax(preds, axis=1))

    acc = accuracy_score(y_true, y_pred)
    fold_acc.append(acc)

    print(f"Fold {fold+1} Accuracy: {acc:.4f}")

    # Track best HP globally
    if acc > best_score:
        best_score = acc
        best_hp_global = best_hp

# ============================================================
# 🔥 FINAL TRAINING ON FULL (TRAIN+VAL)
# ============================================================

final_model = build_mobilenet(best_hp_global)

final_model.fit(
    generator(X_full, y_full),
    steps_per_epoch=len(X_full)//BATCH_SIZE,
    epochs=EPOCHS,
    verbose=1
)

# ============================================================
# 🔥 TEST EVALUATION (UNCHANGED TEST SET)
# ============================================================

y_true, y_pred = [], []
test_gen = generator(X_test, y_test)

for _ in range(len(X_test)//BATCH_SIZE):
    xb, yb = next(test_gen)
    preds = final_model.predict(xb, verbose=0)

    y_true.extend(np.argmax(yb, axis=1))
    y_pred.extend(np.argmax(preds, axis=1))

print("\n🎯 FINAL TEST ACCURACY:", accuracy_score(y_true, y_pred))

print("\n📊 Classification Report:")
print(classification_report(y_true, y_pred, target_names=class_names))

# ============================================================
# 📊 CV RESULT
# ============================================================

print("\nCV Mean Accuracy:", np.mean(fold_acc))
print("CV Std:", np.std(fold_acc))

In [ ]:
# ============================================================
# XCEPTION + BAYESIAN OPT + 5-FOLD CV
# CV ON (TRAIN + VAL), TEST KEPT FIXED
# VANILLA LDM AUGMENTED DATASET
# ============================================================

import os, cv2
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import Xception
from tensorflow.keras.applications.xception import preprocess_input
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report
import keras_tuner as kt

# ---------------- CONFIG ----------------
BASE_DIR = "/kaggle/input/datasets/pankajsthakre/VanillaLDM/BambooLeafDS"

TRAIN_DIR = os.path.join(BASE_DIR, "train")
VAL_DIR   = os.path.join(BASE_DIR, "val")
TEST_DIR  = os.path.join(BASE_DIR, "test")

IMG_SIZE = 224   # ⚠ Xception default is 299, but 224 works for consistency
BATCH_SIZE = 32
EPOCHS = 10
FOLDS = 5
SEED = 42

tf.random.set_seed(SEED)
np.random.seed(SEED)

# ---------------- LOAD FUNCTION ----------------
def load_paths_labels(directory):
    paths, labels = [], []
    class_names = sorted(os.listdir(directory))

    for i, cls in enumerate(class_names):
        cls_dir = os.path.join(directory, cls)
        for f in os.listdir(cls_dir):
            paths.append(os.path.join(cls_dir, f))
            labels.append(i)

    return np.array(paths), np.array(labels), class_names

# ---------------- LOAD DATA ----------------
X_train, y_train, class_names = load_paths_labels(TRAIN_DIR)
X_val, y_val, _ = load_paths_labels(VAL_DIR)
X_test, y_test, _ = load_paths_labels(TEST_DIR)

# 🔥 COMBINE TRAIN + VAL
X_full = np.concatenate([X_train, X_val])
y_full = np.concatenate([y_train, y_val])

NUM_CLASSES = len(class_names)

print("Train+Val:", len(X_full))
print("Test:", len(X_test))

# ---------------- GENERATOR ----------------
def generator(paths, labels):
    while True:
        idx = np.random.permutation(len(paths))

        for i in range(0, len(paths), BATCH_SIZE):
            batch = idx[i:i+BATCH_SIZE]
            imgs, labs = [], []

            for j in batch:
                img = cv2.imread(paths[j])
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
                img = preprocess_input(img.astype("float32"))

                imgs.append(img)
                labs.append(labels[j])

            yield np.array(imgs), tf.keras.utils.to_categorical(labs, NUM_CLASSES)

# ============================================================
# 🔥 MODEL BUILDER (XCEPTION + BAYESIAN)
# ============================================================
def build_xception(hp):

    base = Xception(
        weights="imagenet",
        include_top=False,
        input_shape=(IMG_SIZE, IMG_SIZE, 3)
    )
    base.trainable = False  # Freeze backbone

    x = layers.GlobalAveragePooling2D()(base.output)

    # Tunable Dense
    units = hp.Int("dense_units", 128, 512, step=128)
    x = layers.Dense(units, activation="relu")(x)

    # Tunable Dropout
    dropout = hp.Float("dropout", 0.3, 0.7, step=0.1)
    x = layers.Dropout(dropout)(x)

    outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)

    model = models.Model(base.input, outputs)

    # Tunable LR
    lr = hp.Choice("lr", [1e-3, 1e-4, 1e-5])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(lr),
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

# ============================================================
# 🔥 5-FOLD CV ON TRAIN+VAL
# ============================================================

skf = StratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=SEED)

fold_acc = []
best_hp_global = None
best_score = 0

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_full, y_full)):

    print(f"\n=========== FOLD {fold+1} ===========")

    X_tr, X_vl = X_full[tr_idx], X_full[val_idx]
    y_tr, y_vl = y_full[tr_idx], y_full[val_idx]

    tuner = kt.BayesianOptimization(
        build_xception,
        objective="val_accuracy",
        max_trials=5,
        directory="bayes_xception",
        project_name=f"fold_{fold+1}",
        overwrite=True
    )

    tuner.search(
        generator(X_tr, y_tr),
        steps_per_epoch=len(X_tr)//BATCH_SIZE,
        validation_data=generator(X_vl, y_vl),
        validation_steps=len(X_vl)//BATCH_SIZE,
        epochs=EPOCHS,
        verbose=1
    )

    best_hp = tuner.get_best_hyperparameters(1)[0]
    best_model = tuner.get_best_models(1)[0]

    # ---- validation evaluation ----
    y_true, y_pred = [], []
    val_gen = generator(X_vl, y_vl)

    for _ in range(len(X_vl)//BATCH_SIZE):
        xb, yb = next(val_gen)
        preds = best_model.predict(xb, verbose=0)

        y_true.extend(np.argmax(yb, axis=1))
        y_pred.extend(np.argmax(preds, axis=1))

    acc = accuracy_score(y_true, y_pred)
    fold_acc.append(acc)

    print(f"Fold {fold+1} Accuracy: {acc:.4f}")

    # Track best HP globally
    if acc > best_score:
        best_score = acc
        best_hp_global = best_hp

# ============================================================
# 🔥 FINAL TRAINING ON FULL (TRAIN+VAL)
# ============================================================

final_model = build_xception(best_hp_global)

final_model.fit(
    generator(X_full, y_full),
    steps_per_epoch=len(X_full)//BATCH_SIZE,
    epochs=EPOCHS,
    verbose=1
)

# ============================================================
# 🔥 TEST EVALUATION (UNCHANGED TEST SET)
# ============================================================

y_true, y_pred = [], []
test_gen = generator(X_test, y_test)

for _ in range(len(X_test)//BATCH_SIZE):
    xb, yb = next(test_gen)
    preds = final_model.predict(xb, verbose=0)

    y_true.extend(np.argmax(yb, axis=1))
    y_pred.extend(np.argmax(preds, axis=1))

print("\n🎯 FINAL TEST ACCURACY:", accuracy_score(y_true, y_pred))

print("\n📊 Classification Report:")
print(classification_report(y_true, y_pred, target_names=class_names))

# ============================================================
# 📊 CV RESULT
# ============================================================

print("\nCV Mean Accuracy:", np.mean(fold_acc))
print("CV Std:", np.std(fold_acc))